# Ollama cho chatbot Smart Healthcare (chỉ bật trong buổi demo)

Theo Giải pháp kỹ thuật §11.7. Dịch vụ chatbot chạy trên VPS; notebook này chỉ chạy **model** bằng Ollama trên Colab và mở ra Internet qua **ngrok static domain**, chặn mọi request thiếu hoặc sai header `x-api-key`.

**Chuẩn bị một lần** (Colab → biểu tượng chìa khóa *Secrets*, bật *Notebook access*):

| Secret | Giá trị |
| --- | --- |
| `NGROK_AUTHTOKEN` | authtoken của tài khoản ngrok người demo |
| `NGROK_DOMAIN` | static domain, ví dụ `ten-cua-ban.ngrok-free.app` (không có `https://`) |
| `OLLAMA_API_KEY` | khóa dài ngẫu nhiên; **trùng** `OLLAMA_API_KEY` trong `.env.staging` trên VPS |

Trên VPS: `OLLAMA_BASE_URL=https://<NGROK_DOMAIN>`, `OLLAMA_MODEL` trùng `MODEL` bên dưới.

**Chạy:** Runtime → Change runtime type → GPU (T4), rồi *Run all*. Bật sớm khoảng 30 phút trước demo. Giữ tab mở, laptop cắm sạc, không cho máy ngủ. Sau demo đổi `TAT_SAU_DEMO = True` ở ô cuối rồi chạy riêng ô đó để tắt (*Run all* không tắt).

Không lưu output có khóa: trước khi commit notebook, *Edit → Clear all outputs*. Notebook không in khóa ra màn hình.

In [ ]:
# Phiên bản đã ghim: Dev, Staging, Prod dùng cùng tag (Giải pháp §8.7). Đổi tag chỉ sau khi bộ kịch bản đạt (§11.2).
OLLAMA_VERSION = "0.12.10"
MODEL = "qwen2.5:3b"
OLLAMA_PORT = 11434

In [ ]:
# 1. Cài Ollama đúng phiên bản bằng script cài đặt chính thức.
import os, subprocess
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd", "pciutils"], check=False, stdout=subprocess.DEVNULL)
subprocess.run(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh", shell=True, check=True)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Chạy ollama serve nền: giữ model trong bộ nhớ (KEEP_ALIVE=-1), xử lý một request mỗi lần (Giải pháp §11.7.4).
import time, urllib.request
env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="1")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2)
        print("Ollama đang chạy")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama không khởi động; xem ollama.log")

In [ ]:
# 3. Kéo model đã ghim.
subprocess.run(["ollama", "pull", MODEL], env=env, check=True)

In [ ]:
# 4. Đo tốc độ sinh (Giải pháp §11.7.3 bước 1): khoảng 8 token/giây trở lên là dùng được. Lần đầu gồm cả nạp model.
import json
def generate(prompt, num_predict=60):
    body = json.dumps({"model": MODEL, "prompt": prompt, "stream": False, "options": {"num_predict": num_predict}}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{OLLAMA_PORT}/api/generate", data=body, headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=300))
for attempt in (1, 2):
    r = generate("Đặt lịch khám bác sĩ Nam ngày mai")
    print(f"Lần {attempt}: nạp {r['load_duration']/1e9:.1f}s, sinh {r['eval_count']/(r['eval_duration']/1e9):.1f} token/giây")

In [ ]:
# 5. Đọc secret từ Colab Secrets (không ghi vào notebook, không in ra) và sinh traffic policy của ngrok:
#    request thiếu hoặc sai x-api-key bị trả 401 ngay ở ngrok, không tới Ollama (Giải pháp §11.7.4).
from google.colab import userdata
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN").removeprefix("https://").strip("/")
OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")
assert len(OLLAMA_API_KEY) >= 24, "OLLAMA_API_KEY quá ngắn; dùng chuỗi ngẫu nhiên dài (ví dụ openssl rand -hex 32)"
assert "'" not in OLLAMA_API_KEY, "OLLAMA_API_KEY không được chứa dấu nháy đơn"
policy = f"""on_http_request:
  - expressions:
      - "!('x-api-key' in req.headers) || req.headers['x-api-key'][0] != '{OLLAMA_API_KEY}'"
    actions:
      - type: custom-response
        config:
          status_code: 401
          body: Unauthorized
"""
with open("policy.yml", "w") as f:
    f.write(policy)
os.chmod("policy.yml", 0o600)
print("Đã đọc secret và ghi policy.yml (không in khóa).")

In [ ]:
# 6. Cài ngrok và mở tunnel với static domain. --host-header cần vì Ollama từ chối Host lạ.
subprocess.run("curl -fsSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin", shell=True, check=True)
subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN], check=True, stdout=subprocess.DEVNULL)
ngrok_proc = subprocess.Popen(
    ["ngrok", "http", str(OLLAMA_PORT), f"--url={NGROK_DOMAIN}", f"--host-header=localhost:{OLLAMA_PORT}",
     "--traffic-policy-file=policy.yml", "--log=stdout"],
    stdout=open("ngrok.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
print("ngrok đang chạy" if ngrok_proc.poll() is None else "ngrok đã dừng; xem ngrok.log")

In [ ]:
# 7. Kiểm từ ngoài (Giải pháp §11.7.4 bước 6, §11.7.6): không header → 401; đúng header → /api/tags có tag đã ghim.
import urllib.error
BASE = f"https://{NGROK_DOMAIN}"
def get(path, key=None):
    headers = {"x-api-key": key} if key else {}
    try:
        with urllib.request.urlopen(urllib.request.Request(BASE + path, headers=headers), timeout=15) as r:
            return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, ""
status, _ = get("/api/tags")
print(f"Không header: {status} ({'đúng' if status == 401 else 'SAI: phải là 401'})")
status, body = get("/api/tags", OLLAMA_API_KEY)
has_model = MODEL in body
print(f"Đúng header: {status}, có tag {MODEL}: {has_model}")

In [ ]:
# 8. Làm nóng qua đúng đường ngrok: nạp model và phần đầu prompt. Lần gọi công cụ đầu tiên chậm hơn hẳn các lần sau,
#    nên chạy ô này trước khi mở demo; sau đó từ VPS chạy chatbot-service/scripts/measure_model.py để đo p95.
body = json.dumps({"model": MODEL, "messages": [{"role": "user", "content": "Xin chào"}], "stream": False}).encode()
req = urllib.request.Request(BASE + "/api/chat", data=body, headers={"x-api-key": OLLAMA_API_KEY, "Content-Type": "application/json"})
started = time.time()
json.load(urllib.request.urlopen(req, timeout=300))
print(f"Làm nóng xong sau {time.time() - started:.1f}s. Kiểm GET /health của chatbot trên VPS: available=true.")

In [ ]:
# 9. Sau demo: tắt ngrok và Ollama. /health của chatbot trở về available=false (chatbot hiện tạm ngưng).
#    Run all chạy cả ô này, nên mặc định không tắt: đổi TAT_SAU_DEMO = True rồi chạy riêng ô này.
TAT_SAU_DEMO = False
if TAT_SAU_DEMO:
    for proc in (ngrok_proc, ollama_proc):
        proc.terminate()
    os.remove("policy.yml")
    print("Đã tắt. Nhớ Runtime → Disconnect and delete runtime.")
else:
    print("Chatbot đang mở cho demo. Muốn tắt: đổi TAT_SAU_DEMO = True rồi chạy lại ô này.")